In [ ]:
# Example 1: Basic SQL Parsing and Formatting
import sqlparse
from sqlparse import sql, tokens

# Sample SQL statements for parsing
raw_sql = "select id,name,email from users where age > 18 and status='active' order by name"
complex_sql = "SELECT u.id, u.name, p.title FROM users u LEFT JOIN posts p ON u.id = p.user_id WHERE u.created_at > '2023-01-01' GROUP BY u.id HAVING COUNT(p.id) > 5"

# Basic parsing and formatting
print("Original SQL:")
print(raw_sql)

# Format with default settings
formatted_sql = sqlparse.format(raw_sql, reindent=True, keyword_case='upper')
print("\nFormatted SQL (uppercase keywords):")
print(formatted_sql)

# Format with custom settings
formatted_custom = sqlparse.format(
    complex_sql,
    reindent=True,
    keyword_case='upper',
    identifier_case='lower',
    strip_comments=True,
    use_space_around_operators=True
)
print("\nComplex SQL formatted:")
print(formatted_custom)

# Parse into tokens
parsed = sqlparse.parse(raw_sql)[0]
print(f"\nParsed statement type: {type(parsed)}")
print(f"Statement has {len(parsed.tokens)} tokens")

# Display first few tokens
print("\nFirst 5 tokens:")
for i, token in enumerate(parsed.tokens[:5]):
    print(f"  {i+1}. {token.ttype}: '{token.value.strip()}'")

print("Basic SQL parsing and formatting complete")

In [ ]:
# Example 2: SQL Statement Analysis and Token Extraction
import sqlparse
from sqlparse import sql, tokens as T

def analyze_sql_statement(sql_text):
    """Analyze SQL statement and extract key components"""
    parsed = sqlparse.parse(sql_text)[0]
    
    analysis = {
        'statement_type': None,
        'tables': [],
        'columns': [],
        'keywords': [],
        'operators': [],
        'literals': []
    }
    
    def extract_from_token(token):
        """Recursively extract information from tokens"""
        if token.ttype is T.Keyword:
            keyword = token.value.upper()
            analysis['keywords'].append(keyword)
            
            # Detect statement type
            if keyword in ['SELECT', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP']:
                if not analysis['statement_type']:
                    analysis['statement_type'] = keyword
        
        elif token.ttype is T.Name:
            # Could be table or column name
            analysis['columns'].append(token.value)
        
        elif token.ttype is T.String.Single or token.ttype is T.Literal.String.Single:
            analysis['literals'].append(token.value)
        
        elif token.ttype is T.Operator:
            analysis['operators'].append(token.value)
        
        # Recursively process sub-tokens
        if hasattr(token, 'tokens'):
            for sub_token in token.tokens:
                extract_from_token(sub_token)
    
    # Process all tokens
    for token in parsed.tokens:
        extract_from_token(token)
    
    return analysis

# Test with different SQL statements
test_statements = [
    "SELECT name, email FROM users WHERE age > 25",
    "INSERT INTO products (name, price) VALUES ('Widget', 19.99)",
    "UPDATE users SET status = 'inactive' WHERE last_login < '2023-01-01'",
    "DELETE FROM logs WHERE created_at < '2023-01-01'"
]

print("SQL Statement Analysis:")
for i, stmt in enumerate(test_statements, 1):
    print(f"\n{i}. Statement: {stmt}")
    analysis = analyze_sql_statement(stmt)
    
    print(f"   Type: {analysis['statement_type']}")
    print(f"   Keywords: {list(set(analysis['keywords']))}")
    print(f"   Identifiers: {list(set(analysis['columns']))[:5]}")
    
    if analysis['literals']:
        print(f"   Literals: {analysis['literals'][:3]}")
    
    if analysis['operators']:
        print(f"   Operators: {list(set(analysis['operators']))}")

print("\nSQL statement analysis and token extraction complete")

In [ ]:
# Example 3: SQL Statement Splitting and Batch Processing
import sqlparse

# Multi-statement SQL script
sql_script = """
-- Create users table
CREATE TABLE users (
    id INTEGER PRIMARY KEY,
    name VARCHAR(100),
    email VARCHAR(255) UNIQUE
);

-- Insert sample data
INSERT INTO users (name, email) VALUES ('Alice', 'alice@example.com');
INSERT INTO users (name, email) VALUES ('Bob', 'bob@example.com');

-- Query data
SELECT * FROM users WHERE name LIKE 'A%';

-- Update data
UPDATE users SET name = 'Alice Smith' WHERE email = 'alice@example.com';
"""

def process_sql_batch(script):
    """Split and process a batch of SQL statements"""
    # Split the script into individual statements
    statements = sqlparse.split(script)
    
    processed_statements = []
    
    for i, stmt in enumerate(statements):
        # Skip empty statements
        cleaned_stmt = stmt.strip()
        if not cleaned_stmt or cleaned_stmt.startswith('--'):
            continue
        
        # Parse and format each statement
        formatted = sqlparse.format(
            cleaned_stmt,
            reindent=True,
            keyword_case='upper',
            strip_comments=True
        )
        
        # Analyze statement type
        parsed = sqlparse.parse(cleaned_stmt)[0]
        stmt_type = 'UNKNOWN'
        
        for token in parsed.flatten():
            if token.ttype is sqlparse.tokens.Keyword and token.value.upper() in [
                'SELECT', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER'
            ]:
                stmt_type = token.value.upper()
                break
        
        processed_statements.append({
            'index': len(processed_statements) + 1,
            'type': stmt_type,
            'original': cleaned_stmt,
            'formatted': formatted,
            'length': len(cleaned_stmt)
        })
    
    return processed_statements

# Process the SQL batch
statements = process_sql_batch(sql_script)

print(f"SQL Batch Processing Results:")
print(f"Found {len(statements)} SQL statements")

for stmt in statements:
    print(f"\n{stmt['index']}. {stmt['type']} Statement ({stmt['length']} chars):")
    print(f"   Original: {stmt['original'][:60]}{'...' if len(stmt['original']) > 60 else ''}")
    print(f"   Formatted:")
    
    # Show formatted version with proper indentation
    for line in stmt['formatted'].split('\n'):
        if line.strip():
            print(f"     {line}")

# Summary statistics
statement_types = {}
for stmt in statements:
    stmt_type = stmt['type']
    statement_types[stmt_type] = statement_types.get(stmt_type, 0) + 1

print(f"\nStatement Type Summary:")
for stmt_type, count in statement_types.items():
    print(f"  {stmt_type}: {count} statement{'s' if count != 1 else ''}")

print("SQL statement splitting and batch processing complete")

In [ ]:
# Example 4: Advanced SQL Formatting and Style Enforcement
import sqlparse
from sqlparse import sql, tokens as T

class SQLStyleFormatter:
    """Advanced SQL formatter with customizable style rules"""
    
    def __init__(self):
        self.style_rules = {
            'keyword_case': 'upper',
            'identifier_case': 'lower',
            'function_case': 'upper',
            'reindent': True,
            'strip_comments': False,
            'use_space_around_operators': True,
            'comma_first': False,
            'indent_width': 4
        }
    
    def format_sql(self, sql_text, style='standard'):
        """Format SQL with predefined or custom style"""
        
        if style == 'compact':
            # Compact style - minimal whitespace
            formatted = sqlparse.format(
                sql_text,
                keyword_case='upper',
                strip_whitespace=True,
                reindent=False
            )
        
        elif style == 'expanded':
            # Expanded style - maximum readability
            formatted = sqlparse.format(
                sql_text,
                reindent=True,
                keyword_case='upper',
                identifier_case='lower',
                use_space_around_operators=True,
                indent_tabs=False,
                indent_width=4
            )
        
        elif style == 'database_specific':
            # Database-specific style (e.g., PostgreSQL)
            # Note: 'preserve' is not valid for identifier_case
            # Valid values are: None, 'upper', 'lower', 'capitalize'
            formatted = sqlparse.format(
                sql_text,
                reindent=True,
                keyword_case='upper',
                identifier_case='capitalize',  # Changed from 'preserve' to 'capitalize'
                strip_comments=False
            )
        
        else:  # standard
            # Standard readable style
            formatted = sqlparse.format(
                sql_text,
                **self.style_rules
            )
        
        return formatted
    
    def validate_style(self, sql_text):
        """Check if SQL follows style guidelines"""
        issues = []
        parsed = sqlparse.parse(sql_text)[0]
        
        for token in parsed.flatten():
            if token.ttype is T.Keyword:
                if self.style_rules['keyword_case'] == 'upper' and not token.value.isupper():
                    issues.append(f"Keyword '{token.value}' should be uppercase")
                elif self.style_rules['keyword_case'] == 'lower' and not token.value.islower():
                    issues.append(f"Keyword '{token.value}' should be lowercase")
        
        return issues

# Test different formatting styles
test_sql = """
select u.id,u.name,u.email,p.title,count(c.id) as comment_count from users u 
left join posts p on u.id=p.user_id left join comments c on p.id=c.post_id 
where u.status='active' and p.published_at>'2023-01-01' 
group by u.id,u.name,u.email,p.title having count(c.id)>5 
order by comment_count desc,u.name asc limit 10
"""

formatter = SQLStyleFormatter()

print("SQL Style Formatting Demonstration:")
print("\nOriginal SQL:")
print(test_sql.strip())

styles = ['standard', 'compact', 'expanded', 'database_specific']

for style in styles:
    print(f"\n{'-'*20} {style.upper()} STYLE {'-'*20}")
    formatted = formatter.format_sql(test_sql, style=style)
    print(formatted)

# Style validation
print(f"\n{'-'*20} STYLE VALIDATION {'-'*20}")
messy_sql = "select name from users where age > 18"
issues = formatter.validate_style(messy_sql)

print(f"Checking: {messy_sql}")
if issues:
    print("Style issues found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("No style issues found")

# Show corrected version
corrected = formatter.format_sql(messy_sql)
print(f"\nCorrected: {corrected.strip()}")

print("Advanced SQL formatting and style enforcement complete")

In [ ]:
# Example 5: Production SQL Analysis and Optimization Tools
import sqlparse
from sqlparse import sql, tokens as T
from collections import defaultdict, Counter
import re

class SQLAnalyzer:
    """Production-ready SQL analysis and optimization toolkit"""
    
    def __init__(self):
        self.metrics = defaultdict(int)
        self.tables_accessed = set()
        self.query_patterns = Counter()
    
    def analyze_query_complexity(self, sql_text):
        """Analyze query complexity and potential issues"""
        parsed = sqlparse.parse(sql_text)[0]
        
        complexity_score = 0
        complexity_factors = []
        
        # Count different SQL elements
        joins = 0
        subqueries = 0
        functions = 0
        conditions = 0
        
        sql_upper = sql_text.upper()
        
        # Detect joins
        join_patterns = ['JOIN', 'LEFT JOIN', 'RIGHT JOIN', 'INNER JOIN', 'OUTER JOIN']
        for pattern in join_patterns:
            joins += sql_upper.count(pattern)
        
        # Detect subqueries
        subqueries = sql_upper.count('(SELECT')
        
        # Detect common functions
        function_patterns = ['COUNT(', 'SUM(', 'AVG(', 'MAX(', 'MIN(', 'GROUP_CONCAT(']
        for pattern in function_patterns:
            functions += sql_upper.count(pattern)
        
        # Detect conditions
        conditions = sql_upper.count('WHERE') + sql_upper.count('HAVING')
        
        # Calculate complexity score
        complexity_score = joins * 2 + subqueries * 3 + functions + conditions
        
        if joins > 0:
            complexity_factors.append(f"{joins} joins")
        if subqueries > 0:
            complexity_factors.append(f"{subqueries} subqueries")
        if functions > 0:
            complexity_factors.append(f"{functions} aggregate functions")
        if conditions > 0:
            complexity_factors.append(f"{conditions} conditional clauses")
        
        return {
            'complexity_score': complexity_score,
            'complexity_level': self._get_complexity_level(complexity_score),
            'factors': complexity_factors,
            'joins': joins,
            'subqueries': subqueries,
            'functions': functions,
            'conditions': conditions
        }
    
    def _get_complexity_level(self, score):
        """Determine complexity level based on score"""
        if score <= 2:
            return 'Simple'
        elif score <= 5:
            return 'Moderate'
        elif score <= 10:
            return 'Complex'
        else:
            return 'Very Complex'
    
    def extract_table_references(self, sql_text):
        """Extract table names and their usage patterns"""
        # Simple pattern matching for table extraction
        # Note: This is a basic implementation; production tools would be more sophisticated
        
        tables = set()
        
        # Pattern for FROM clauses
        from_pattern = r'FROM\s+([\w\.]+)'
        tables.update(re.findall(from_pattern, sql_text, re.IGNORECASE))
        
        # Pattern for JOIN clauses
        join_pattern = r'JOIN\s+([\w\.]+)'
        tables.update(re.findall(join_pattern, sql_text, re.IGNORECASE))
        
        # Pattern for INSERT/UPDATE/DELETE
        dml_pattern = r'(?:INSERT\s+INTO|UPDATE|DELETE\s+FROM)\s+([\w\.]+)'
        tables.update(re.findall(dml_pattern, sql_text, re.IGNORECASE))
        
        return list(tables)
    
    def suggest_optimizations(self, sql_text, complexity_analysis):
        """Suggest potential optimizations based on analysis"""
        suggestions = []
        
        if complexity_analysis['complexity_level'] in ['Complex', 'Very Complex']:
            suggestions.append("Consider breaking down complex query into smaller parts")
        
        if complexity_analysis['joins'] > 3:
            suggestions.append("Review join order and consider adding indexes on join columns")
        
        if complexity_analysis['subqueries'] > 2:
            suggestions.append("Consider converting subqueries to JOINs for better performance")
        
        if 'SELECT *' in sql_text.upper():
            suggestions.append("Avoid SELECT * - specify only needed columns")
        
        if 'ORDER BY' in sql_text.upper() and 'LIMIT' not in sql_text.upper():
            suggestions.append("Consider adding LIMIT clause when using ORDER BY")
        
        if not re.search(r'WHERE', sql_text, re.IGNORECASE) and 'SELECT' in sql_text.upper():
            suggestions.append("Consider adding WHERE clause to limit result set")
        
        return suggestions
    
    def generate_analysis_report(self, sql_queries):
        """Generate comprehensive analysis report"""
        report = {
            'total_queries': len(sql_queries),
            'complexity_distribution': Counter(),
            'tables_accessed': set(),
            'query_types': Counter(),
            'optimization_suggestions': []
        }
        
        for i, query in enumerate(sql_queries):
            # Analyze each query
            complexity = self.analyze_query_complexity(query)
            tables = self.extract_table_references(query)
            suggestions = self.suggest_optimizations(query, complexity)
            
            # Update report data
            report['complexity_distribution'][complexity['complexity_level']] += 1
            report['tables_accessed'].update(tables)
            report['optimization_suggestions'].extend([
                f"Query {i+1}: {suggestion}" for suggestion in suggestions
            ])
            
            # Detect query type
            query_upper = query.upper().strip()
            if query_upper.startswith('SELECT'):
                report['query_types']['SELECT'] += 1
            elif query_upper.startswith('INSERT'):
                report['query_types']['INSERT'] += 1
            elif query_upper.startswith('UPDATE'):
                report['query_types']['UPDATE'] += 1
            elif query_upper.startswith('DELETE'):
                report['query_types']['DELETE'] += 1
        
        return report

# Demonstrate production SQL analysis
analyzer = SQLAnalyzer()

# Sample production queries
production_queries = [
    "SELECT * FROM users WHERE created_at > '2023-01-01'",
    "SELECT u.name, COUNT(p.id) FROM users u LEFT JOIN posts p ON u.id = p.user_id GROUP BY u.id ORDER BY COUNT(p.id) DESC",
    "UPDATE users SET last_login = NOW() WHERE id = 123",
    "SELECT u.id, u.name, p.title, (SELECT COUNT(*) FROM comments c WHERE c.post_id = p.id) as comment_count FROM users u JOIN posts p ON u.id = p.user_id WHERE u.status = 'active' AND p.published_at > '2023-01-01'",
    "DELETE FROM logs WHERE created_at < DATE_SUB(NOW(), INTERVAL 30 DAY)"
]

print("Production SQL Analysis Report:")
print("=" * 50)

# Analyze individual queries
for i, query in enumerate(production_queries, 1):
    print(f"\nQuery {i}:")
    print(f"SQL: {query[:80]}{'...' if len(query) > 80 else ''}")
    
    complexity = analyzer.analyze_query_complexity(query)
    tables = analyzer.extract_table_references(query)
    suggestions = analyzer.suggest_optimizations(query, complexity)
    
    print(f"Complexity: {complexity['complexity_level']} (Score: {complexity['complexity_score']})")
    print(f"Tables: {', '.join(tables) if tables else 'None detected'}")
    
    if suggestions:
        print(f"Suggestions: {'; '.join(suggestions)}")

# Generate overall report
report = analyzer.generate_analysis_report(production_queries)

print(f"\n{'='*50}")
print("OVERALL ANALYSIS SUMMARY")
print(f"{'='*50}")
print(f"Total queries analyzed: {report['total_queries']}")
print(f"Query types: {dict(report['query_types'])}")
print(f"Complexity distribution: {dict(report['complexity_distribution'])}")
print(f"Tables accessed: {', '.join(sorted(report['tables_accessed']))}")
print(f"Total optimization suggestions: {len(report['optimization_suggestions'])}")

print("Production SQL analysis and optimization tools complete")